# 18장 · 분석법 선택의 나무

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [18장 · 분석법 선택의 나무](https://grow.minds.kr/textbooks/css-methods/causal/book/ch18-분석법-선택의-나무.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 같은 물음, 두 경로
평균을 직접 빼는 것과 집단 표시를 넣은 회귀는 **같은 수**를 낸다. 소수 넷째 자리까지 같다.

In [ ]:
exp = load("exp")
a, b = exp[exp.cond == 1].mil.values, exp[exp.cond == 0].mil.values
print("① 평균을 직접 뺀다:", round(a.mean() - b.mean(), 4))            # 0.2828
bb, se, p, r2 = ols(exp.mil, [exp.cond])
print("② 회귀 계수:", round(bb[1], 4), " t:", round(bb[1]/se[1], 3), " p:", round(p[1], 3))

## 3. 셋째 경로도 같은 곳으로
뒤섞기는 공식을 안 쓰고 같은 결론에 닿는다(.029 대 .028).

In [ ]:
rng = np.random.default_rng(73)
pool = np.concatenate([a, b]); null = []
for _ in range(10000):
    q = rng.permutation(pool)
    null.append(q[:len(a)].mean() - q[len(a):].mean())
print("뒤섞기 p =", round(float(np.mean(np.abs(null) >= abs(a.mean()-b.mean()))), 4))  # 0.0292

## 4. 짝을 지었는지가 답을 바꾼다
같은 사람의 1차·3차를 **짝지어** 보면 t = -2.21, 남남으로 보면 -1.64. 자료가 같아도 설계가 다르면 검정이 다르다.

In [ ]:
w = load("panel").pivot(index="id", columns="wave", values="mil").dropna()
x1, x3 = w[1].values, w[3].values; d = x3 - x1
print(len(d), round(d.mean(), 3), round(np.corrcoef(x1, x3)[0, 1], 3))   # 350 -0.13 0.453
print("짝지어:", round(d.mean() / (d.std(ddof=1)/np.sqrt(len(d))), 2))    # -2.21
sp = np.sqrt((x1.var(ddof=1) + x3.var(ddof=1)) / 2)
print("남남으로:", round((x3.mean()-x1.mean()) / (sp*np.sqrt(2/len(d))), 2))  # -1.64

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.